In [5]:
import random
import numpy as np
from tqdm import tqdm
import math
import matplotlib.pyplot as plt
from copy import deepcopy
import gymnasium as gym

pygame 2.5.1 (SDL 2.28.2, Python 3.11.4)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [1]:
class Maze:
    def __init__(self):
        self.maze = [
            "OOOOOOOCG",
            "OOCOOOOCO",
            "SOCOOOOCO",
            "OOCOOOOOO",
            "OOOOOCOOO",
            "OOOOOOOOO",
        ]
        self.starting_state = (2,0)
        self.goal_state = (0, 8)

        self.number_of_steps = 0
        self.current_state = self.starting_state

        self.terminated = 0
        self.truncated = 0

        self.terminal_states = [(1,2), (2, 2), (3, 2), (0, 7), (1, 7), (2, 7), (4, 5), (0,8)]

    def get_action_space(self):
        return ["up", "down", "left", "right"]

    def print_maze(self):
        print(self.maze)

    def change_state(self, action):

        new_state = (0, 0)

        if action == "up":
            if self.current_state[0] == 0:
                return
            new_state = (self.current_state[0] - 1, self.current_state[1])
        elif action == "down":
            if self.current_state[0] == 5:
                return
            new_state = (self.current_state[0] + 1, self.current_state[1])
        elif action == "left":
            if self.current_state[1] == 0:
                return
            new_state = (self.current_state[0], self.current_state[1] - 1)
        elif action == "right":
            if self.current_state[1] == 8:
                return

            new_state = (self.current_state[0], self.current_state[1] + 1)

        if (
            self.maze[new_state[0]][new_state[1]] == "O"
            or self.maze[new_state[0]][new_state[1]] == "S"
        ):
            self.current_state = new_state
        elif self.maze[new_state[0]][new_state[1]] == "G":
            self.current_state = new_state

    def step(self, action):

        if self.terminated or self.truncated:
            observation = self.current_state
            reward = 0
            return observation, reward, self.terminated, self.truncated

        self.number_of_steps += 1

        self.change_state(action)

        observation = self.current_state

        reward = 0
        if self.maze[self.current_state[0]][self.current_state[1]] == "G":
            reward = 1

            self.terminated  = 1
        else:
            reward = 0

        if self.number_of_steps == 1000:
            self.truncated = 1

        return observation, reward, self.terminated, self.truncated

    def reset(self):
        self.number_of_steps = 0
        self.current_state = self.starting_state

        self.terminated = 0
        self.truncated = 0

In [2]:
class ChangingMaze(Maze):
    def __init__(self):

        self.maze = [
            "OOOOOOOOG",
            "OOOOOOOOO",
            "OOOOOOOOO",
            "CCCCCCCCO",
            "OOOOOOOOO",
            "OOOSOOOOO",
        ]

        self.starting_state = (5,3)
        self.goal_state = (0, 8)

        self.number_of_steps = 0
        self.current_state = self.starting_state

        self.terminated = 0
        self.truncated = 0

        self.terminal_states = [
            (3, 0),
            (3, 1),
            (3, 2),
            (3, 3),
            (3, 4),
            (3, 5),
            (3, 6),
            (3, 7),
            (0, 8)
        ]

        self.total_number_of_steps = 0
    def reset(self):
        self.total_number_of_steps += self.number_of_steps

        if self.total_number_of_steps > 1000:
            self.maze = [
                "OOOOOOOOG",
                "OOOOOOOOO",
                "OOOOOOOOO",
                "OCCCCCCCC",
                "OOOOOOOOO",
                "OOOSOOOOO",
            ]

            self.terminal_states = [
                (3, 1),
                (3, 2),
                (3, 3),
                (3, 4),
                (3, 5),
                (3, 6),
                (3, 7),
                (3, 8),
                (0, 8)
            ]

        self.number_of_steps = 0
        self.current_state = self.starting_state

        self.terminated = 0
        self.truncated = 0

In [3]:
class Dyna_Q:
    def __init__(self, env, n, number_of_episodes = 100):
        self.states_and_actions_taken = {}
        self.env = env
        self.n = n

        self.Q = {}
        self.Model = {}

        self.initialize_Q_and_Model()

        self.number_of_episodes = number_of_episodes

        self.trajectories = []

    def initialize_Q_and_Model(self):

        for row in range(6):
            for col in range(9):
                self.Q[(row, col)] = {"up": 0, "down": 0, "left": 0, "right": 0}
                self.Model[(row, col)] = {"up": (0,0), "down": (0,0), "left": (0,0), "right": (0,0)}

    def get_argmax(self, S):
        state_dict = self.Q[S]

        max_value = max(state_dict.values())
        return [action for action, value in state_dict.items() if value == max_value]

    def get_max(self, S):
        state_dict = self.Q[S]

        action = max(state_dict, key=state_dict.get)
        return self.Q[S][action]

    def get_action(self, epsilon, S):
        action_space = [0, 1, 2, 3]
        action = random.choice(action_space)
        if random.random() > epsilon:
            possible_actions = self.get_argmax(S)
            action = random.choice(possible_actions)

        return action

    def update_state_action_collection(self, S, A):
        if S not in self.states_and_actions_taken:
            self.states_and_actions_taken[S] = []

        if A not in self.states_and_actions_taken[S]:
            self.states_and_actions_taken[S].append(A)

    def get_policy(self):
        policy = {}
        terminal_states = [5, 7, 11, 12, 15]
        for S in self.Q:

            if S in terminal_states:
                policy[S] = "Terminal"
                continue

            A = self.get_argmax(S)[0]

            policy[S] = A
        return policy

    def do_planning_updates(self):
        for _ in range(self.n):

            S = random.choice(list(self.states_and_actions_taken.keys()))
            A = random.choice(self.states_and_actions_taken[S])

            R, S_prime = self.Model[S][A]

            self.Q[S][A] = self.Q[S][A] + 0.1 * (
                R + 0.9999999999999999 * self.get_max(S_prime) - self.Q[S][A]
            )

    def do_one_dyna_Q_iteration(self):
        self.env.reset()

        S = self.env.current_state
        R = 0
        S_prime = self.env.current_state
        terminated = 0
        truncated = 0

        trajectory = []

        while not terminated and not truncated:
            S = S_prime
            trajectory.append(S)
            A = self.get_action(0.8, S)

            self.update_state_action_collection(S, A)

            S_prime, R, terminated, truncated  = self.env.step(A)

            self.Q[S][A] = self.Q[S][A] + 0.1 * (
                R + 0.9999999999999999 * self.get_max(S_prime) - self.Q[S][A]
            )

            self.Model[S][A] = (R, S_prime)

            self.do_planning_updates()

        self.trajectories.append(trajectory)

    def do_dyna_Q(self):
        for i in tqdm(range(self.number_of_episodes)):
            self.do_one_dyna_Q_iteration()

In [4]:
class Dyna_Q_plus(Dyna_Q):

    def __init__(
        self,
        env,
        n,
        κ,
        number_of_episodes=100
    ):
        super().__init__(env, n, number_of_episodes)

        self.κ = κ

    def initialize_Q_and_Model(self):

        for row in range(6):
            for col in range(9):
                self.Q[(row, col)] = {"up": 0, "down": 0, "left": 0, "right": 0}
                self.Model[(row, col)] = {
                    "up": [0, 0, 0],
                    "down": [0, 0, 0],
                    "left": [0, 0, 0],
                    "right": [0, 0, 0],
                }


    def get_action(self, S):

        state_dict = deepcopy(self.Q[S])

        for A in state_dict.keys():

            τ = self.Model[S][A][2]
            state_dict[A] += self.κ * math.sqrt(τ)


        max_value = max(state_dict.values())
        best_actions = [action for action, value in state_dict.items() if value == max_value]



        action = random.choice(best_actions)



        return action

    def do_planning_updates(self):
        for _ in range(self.n):

            S = random.choice(list(self.states_and_actions_taken.keys()))
            A = random.choice(self.states_and_actions_taken[S])

            R, S_prime, _ = self.Model[S][A]

            self.Q[S][A] = self.Q[S][A] + 0.1 * (
                R + 0.9999999999999999 * self.get_max(S_prime) - self.Q[S][A]
            )
            # R + self.κ*math.sqrt(τ) +

    def increase_τ(self):
        for row in range(6):
            for col in range(9):
                self.Model[(row, col)]["up"][2] += 1
                self.Model[(row, col)]["down"][2] += 1
                self.Model[(row, col)]["left"][2] += 1
                self.Model[(row, col)]["right"][2] += 1

    def do_one_dyna_Q_iteration(self):
        self.env.reset()

        S = self.env.current_state
        R = 0
        S_prime = self.env.current_state
        terminated = 0
        truncated = 0

        trajectory = []

        while not terminated and not truncated:
            S = S_prime
            trajectory.append(S)
            A = self.get_action(S)

            self.update_state_action_collection(S, A)

            S_prime, R, terminated, truncated = self.env.step(A)


            self.Q[S][A] = self.Q[S][A] + 0.1 * (
                R + 0.9999999999999999 * self.get_max(S_prime) - self.Q[S][A]
            )

            self.increase_τ()
            self.Model[S][A] = [R, S_prime, 0]

            self.do_planning_updates()

        self.trajectories.append(trajectory)

In [452]:
maze = Maze()

In [254]:
dyna_q = Dyna_Q_plus(maze, 5, 0.1 ,40)

In [453]:
dyna_q = Dyna_Q(maze, 5, 100)

In [454]:
dyna_q.do_dyna_Q()

100%|██████████| 100/100 [00:00<00:00, 127.96it/s]


In [7]:
class PlotPolicy():
    def __init__(self, policy):
        self.policy = policy

    def transform_policy(self):
        np_policy = np.array([self.policy[key] for key in self.policy]).reshape(6, 9)
        np_policy = np.where(np_policy == "left", 0, np_policy)
        np_policy = np.where(np_policy == "down", 1, np_policy)
        np_policy = np.where(np_policy == "right", 2, np_policy)
        np_policy = np.where(np_policy == "up", 3, np_policy)
        np_policy = np.where(np_policy == "Terminal", -1, np_policy)


        self.policy = np_policy.astype(float)

    def visualize_policy(self):
        self.transform_policy()

        fig, ax = plt.subplots()
        ax.matshow(self.policy, cmap='Blues')

        for i in range(len(self.policy)):
            for j in range(len(self.policy[0])):
                ax.text(j, i, self.policy[i][j], ha='center', va='center')

        plt.xticks([])
        plt.yticks([])
        plt.show()


So, Dyna Q+ isn't really working. Instead, what you want to do is that you want to interact with the environment in such a way that you should explore more the states that haven't really been touched in a while. But, still keep to your premises.

It's really hard to make Dyna Q+ work. Especially in non stationary environments. I feel the best strategy is continously explore. Also, the actions you haven't really tried, perhaps we could define a probability distribution to checek whether those actions could be needing some more exploration. I mean there should be less certainty in the actions that haven't been tried. 

Thing is that state action pairs that haven't been explored that much should have a higher propensity of getting explored but it seems that in such an exploration it isn't really randomized enough to give everything a fair chance in exploring properly.

In [8]:
class Model():
    def __init__(self):
        self.model = {}

    def get_states(self):
        state_action_pairs =  list(self.model.keys())
        return list(set(list(map(lambda x: x[0], state_action_pairs ))))

    def get_state_action_pairs(self):
        return list(self.model.keys())

    def get_actions(self, state):
        state_action_pairs = list(self.model.keys())
        actions = [A for S, A in state_action_pairs if S == state]
        return actions

    def handle_observed_state_action_pair(self, S, A, R, S_prime):
        if (R, S_prime) not in self.model[(S, A)]:
            self.model[(S, A)][(R, S_prime)] = 1
        else:
            self.model[(S, A)][(R, S_prime)] += 1

    def add_transition(self, S, A, R, S_prime):

        if (S, A) not in self.model:
            self.model[(S, A)] = {}
            self.model[(S, A)][(R, S_prime)] = 1
        else:
            self.handle_observed_state_action_pair(S, A, R, S_prime)

    def get_probability_distribution(self, S, A):
        rewards_next_states_distribution = deepcopy(self.model[(S, A)])

        reward_next_states_probability_distribution = {}

        for (R, S_prime), count in rewards_next_states_distribution.items():
            reward_next_states_probability_distribution[(R, S_prime)] = count / sum(rewards_next_states_distribution.values())

        return reward_next_states_probability_distribution

    def select_random_R_S_prime(self, S, A):
        rewards_next_states_distribution = self.get_probability_distribution(S, A)

        keys = list(rewards_next_states_distribution.keys())
        weights = list(rewards_next_states_distribution.values())

        return random.choices(keys, weights, k=1)[0]

In [10]:
class Dyna_Q_stochastic(Dyna_Q):
    def __init__(self, env, n, number_of_episodes=100):
        super().__init__(env, n, number_of_episodes)


        self.model = Model()

    def initialize_Q_and_Model(self):

        # for row in range(6):
        #     for col in range(9):

        for state in range(16):
            self.Q[state] = {0: 0, 1: 0, 2: 0, 3: 0}


    def do_planning_updates(self):
        for _ in range(self.n):

            S = random.choice(self.model.get_states())
            A = random.choice(self.model.get_actions(S))

            R, S_prime = self.model.select_random_R_S_prime(S, A)

            self.Q[S][A] = self.Q[S][A] + 0.1 * (
                R + 1 * self.get_max(S_prime) - self.Q[S][A]
            )

    def do_one_dyna_Q_iteration(self):
        self.env.reset()

        # S = self.env.current_state
        S = 0
        R = 0
        S_prime = 0
        # S_prime = self.env.current_state
        terminated = 0
        truncated = 0

        trajectory = []

        while not terminated and not truncated:
            S = S_prime
            trajectory.append(S)
            A = self.get_action(0.2, S)

            S_prime, R, terminated, truncated, _ = self.env.step(A)

            self.Q[S][A] = self.Q[S][A] + 0.1 * (
                R + 1 * self.get_max(S_prime) - self.Q[S][A]
            )


            self.model.add_transition(S, A, R, S_prime)

            self.do_planning_updates()

        self.trajectories.append(trajectory)

The idea for Dyna Q Plus Stochastic would be that we would need to combine it with optimistic initalization so that it explores paths that it hasn't quite tried before and also you would need to increase the tau variable for every one of the state action pairs that you have tried.

In [165]:
class Model_plus(Model):
    def __init__(self):
        super().__init__()

        self.τ = {}

        # Our Model are two nested dictionaries with the following structure: The first dictionary tracks the state and action pair, each state action pair has a dictionary that tracks the reward and next state pair and each reward and next state pair has a list that tracks the number of occurrences of that particular reward and next state and the last time it was observed

    def increase_τ(self):
        for S, A in self.τ.keys():
            self.τ[(S, A)] += 1

    def get_τ(self, S, A):
        if (S, A) not in self.τ:
            self.τ[(S, A)] = 0

        return self.τ[(S, A)]

    def add_transition(self, S, A, R, S_prime):

        self.increase_τ()
        self.τ[(S, A)] = 0

        if (S, A) not in self.model:
            self.model[(S, A)] = {}
            self.model[(S, A)][(R, S_prime)] = 1
        else:
            self.handle_observed_state_action_pair(S, A, R, S_prime)


In [193]:
class Dyna_Q_plus_stochastic(Dyna_Q_stochastic):
    def __init__(self, env, n, κ, number_of_episodes=100):

        super().__init__(env, n, number_of_episodes)

        self.model = Model_plus()
        self.κ = κ

    def initialize_Q_and_Model(self):

        # We will use optimistic initialization so that it tries to explore the actions that it hasn't tried before.

        terminal_states = [5, 7, 11, 12, 15]

        for state in range(16):
            if state in terminal_states:
                self.Q[state] = {0: 0, 1: 0, 2: 0, 3: 0}
            else:
                self.Q[state] = {0: 1, 1: 1, 2: 1, 3: 1}

    def get_action(self, S):

        state_dict = deepcopy(self.Q[S])

        for A in state_dict.keys():

            τ = self.model.get_τ(S, A)
            state_dict[A] += self.κ * math.sqrt(τ)

        max_value = max(state_dict.values())
        best_actions = [
            action for action, value in state_dict.items() if value == max_value
        ]

        # print(best_actions)
        action = random.choice(best_actions)

        return action

    def do_one_dyna_Q_iteration(self):
        self.env.reset()

        # S = self.env.current_state
        S = 0
        R = 0
        S_prime = 0
        # S_prime = self.env.current_state
        terminated = 0
        truncated = 0

        trajectory = []

        while not terminated and not truncated:
            S = S_prime
            trajectory.append(S)
            A = self.get_action(S)

            S_prime, R, terminated, truncated, _ = self.env.step(A)

            self.Q[S][A] = self.Q[S][A] + 0.1 * (
                R + 1 * self.get_max(S_prime) - self.Q[S][A]
            )

            self.model.add_transition(S, A, R, S_prime)

            self.do_planning_updates()

        self.trajectories.append(trajectory)

In [194]:
env = gym.make("FrozenLake-v1", is_slippery=True, map_name="4x4")

In [195]:
dyna_q_stochastic = Dyna_Q_plus_stochastic(env, 5, 0.1, 1000)

In [196]:
dyna_q_stochastic.do_dyna_Q()

  0%|          | 0/1000 [00:00<?, ?it/s]

100%|██████████| 1000/1000 [00:03<00:00, 266.40it/s]


In [498]:
np.array(
    [
        [0.82352941, 0.82352941, 0.82352941, 0.82352941],
        [0.54901961, 0.54901961, 0.54901961, 0.82352941],
        [0.72549019, 0.72549019, 0.72549019, 0.82352941],
        [0.54901961, 0.54901961, 0.54901961, 0.82352941],
        [0.82352941, 0.54901961, 0.54901961, 0.54901961],
        [0.0, 0.0, 0.0, 0.0],
        [0.52941176, 0.25490196, 0.52941176, 0.2745098],
        [0.0, 0.0, 0.0, 0.0],
        [0.54901961, 0.54901961, 0.54901961, 0.82352941],
        [0.56862745, 0.82352941, 0.54901961, 0.52941176],
        [0.76470588, 0.58823529, 0.49019608, 0.45098039],
        [0.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0],
        [0.56862745, 0.60784314, 0.88235294, 0.58823529],
        [0.8627451, 0.94117647, 0.90196078, 0.88235294],
        [0.0, 0.0, 0.0, 0.0],
    ]
)

array([[0.82352941, 0.82352941, 0.82352941, 0.82352941],
       [0.54901961, 0.54901961, 0.54901961, 0.82352941],
       [0.72549019, 0.72549019, 0.72549019, 0.82352941],
       [0.54901961, 0.54901961, 0.54901961, 0.82352941],
       [0.82352941, 0.54901961, 0.54901961, 0.54901961],
       [0.        , 0.        , 0.        , 0.        ],
       [0.52941176, 0.25490196, 0.52941176, 0.2745098 ],
       [0.        , 0.        , 0.        , 0.        ],
       [0.54901961, 0.54901961, 0.54901961, 0.82352941],
       [0.56862745, 0.82352941, 0.54901961, 0.52941176],
       [0.76470588, 0.58823529, 0.49019608, 0.45098039],
       [0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        ],
       [0.56862745, 0.60784314, 0.88235294, 0.58823529],
       [0.8627451 , 0.94117647, 0.90196078, 0.88235294],
       [0.        , 0.        , 0.        , 0.        ]])

In [197]:
dyna_q_stochastic.Q

{0: {0: 0.9919031868308326,
  1: 0.9948839773629581,
  2: 0.9937280946952529,
  3: 1.0},
 1: {0: 0.6769738166641025,
  1: 0.7589298130306413,
  2: 0.6670099021526058,
  3: 1.0},
 2: {0: 0.9111853245894465,
  1: 0.923637833006883,
  2: 0.8431525580176399,
  3: 1.0},
 3: {0: 0.6962631479226474,
  1: 0.6969328622332686,
  2: 0.7897294841262342,
  3: 1.0},
 4: {0: 0.9762906064611704,
  1: 0.738178192684252,
  2: 0.47507859835340815,
  3: 0.5374932317501031},
 5: {0: 0, 1: 0, 2: 0, 3: 0},
 6: {0: 0.6165612812490171,
  1: 0.3643205688159304,
  2: 0.4040774531095165,
  3: 0.462602789485552},
 7: {0: 0, 1: 0, 2: 0, 3: 0},
 8: {0: 0.4689756644015113,
  1: 0.5883762465952976,
  2: 0.4788220004737722,
  3: 0.9414369579939508},
 9: {0: 0.6070115866555608,
  1: 0.8947793510560953,
  2: 0.5931951482092103,
  3: 0.5338248209620549},
 10: {0: 0.833000210867485,
  1: 0.5051981869797779,
  2: 0.5421333443002861,
  3: 0.5678293510002437},
 11: {0: 0, 1: 0, 2: 0, 3: 0},
 12: {0: 0, 1: 0, 2: 0, 3: 0},
 13:

In [199]:
dyna_q_stochastic.get_policy()

{0: 3,
 1: 3,
 2: 3,
 3: 3,
 4: 0,
 5: 'Terminal',
 6: 0,
 7: 'Terminal',
 8: 3,
 9: 1,
 10: 0,
 11: 'Terminal',
 12: 'Terminal',
 13: 2,
 14: 1,
 15: 'Terminal'}

In [198]:
dyna_q_stochastic.model.model[(0,0)]


{(0.0, 4): 298, (0.0, 0): 577}

In [550]:
dyna_q_stochastic.model.model[(0, 3)]

{(0.0, 1): 1655, (0.0, 0): 3420}

In [ ]:
π = [0, 3, 3, 3, 0, -1, 0, -1, 3, 1, 0, -1, -1, 2, 1, -2]

In [200]:
dyna_q_stochastic.get_policy().values()

dict_values([3, 3, 3, 3, 0, 'Terminal', 0, 'Terminal', 3, 1, 0, 'Terminal', 'Terminal', 2, 1, 'Terminal'])